# gold_receita

Camada **Gold** da receita: responde ao Henrique (Diretor Comercial) e é a base do diagnóstico da receita.

- **Entrada (temporária):** `silver.tb_faturas`, `silver.tb_leituras` e `silver.tb_unidades_consumidoras`. Quando o `gold_base` existir, a célula de leitura passa a usar `gold.fato_uc_mes` e `gold.dim_uc`.
- **Saída:** `gold.agg_receita_mensal` e `gold.agg_preco_volume`.
- **Alimenta:** `GET /dashboard` (receita, consumo e preço × volume), agente de IA e diagnóstico em `docs/`.
- Segue o documento *Decisões da Engenharia de Dados: camada Gold*. Idempotente: grava em overwrite e pode ser reexecutado.

| Tabela | Grão (1 linha por) | Colunas |
|---|---|---|
| `agg_receita_mensal` | mês × município × bairro × classe | `receita_total`, `receita_energia`, `tributos`, `consumo_kwh`, `faturas`, `tarifa_media` |
| `agg_preco_volume` | município × bairro × classe (P1 × P2) | receita, kWh e tarifa de cada período, `variacao_receita`, `efeito_volume`, `efeito_preco` |

**Métricas**
- `receita_total` = Σ `valor_total`; `receita_energia` = Σ `valor_energia` (sem tributos); `consumo_kwh` = Σ `consumo_kwh` da leitura da mesma UC e mês.
- `tarifa_media` = receita de energia ÷ kWh (nula se o kWh for 0).
- **Preço × volume** (por segmento, depois somado): `efeito_volume` = (kWh₂ − kWh₁) × tarifa₁ e `efeito_preco` = Δ receita de energia − efeito-volume, que é o mesmo que (tarifa₂ − tarifa₁) × kWh₂ e fecha sem resíduo.
- **P1** = primeiro ano da base (2022-07 a 2023-06); **P2** = últimos 12 meses (2025-07 a 2026-06), como na fala do Henrique.
- O kWh usa o `consumo_kwh` da Silver como está (já com a correção da 1ª leitura pós-troca). Assim a Gold reproduz o case: receita +8% e volume −4%.

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

In [0]:
# ---------- Regras de negócio: Receita ----------
SEGMENTO = ["municipio", "bairro", "classe"]
P1 = ("2022-07", "2023-06")      # primeiro ano da base
P2 = ("2025-07", "2026-06")      # últimos 12 meses

In [0]:
# Base UC × mês. Temporário: trocar por gold.fato_uc_mes + gold.dim_uc quando o gold_base existir
ft = spark.table(f"{CATALOGO}.silver.tb_faturas").select(
    "id_uc", "mes_referencia", "valor_total", "valor_energia", "tributos")
lt = spark.table(f"{CATALOGO}.silver.tb_leituras").select("id_uc", "mes_referencia", "consumo_kwh")
uc = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras").select("id_uc", *SEGMENTO)

base = ft.join(lt, ["id_uc", "mes_referencia"], "left").join(uc, "id_uc")
checar_unicidade(base, ["id_uc", "mes_referencia"], "base_receita")   # leitura repetida duplicaria a receita

In [0]:
# agg_receita_mensal: 1 linha por mês × município × bairro × classe
dec = lambda c: F.col(c).cast("decimal(14,2)")
receita = (base.groupBy("mes_referencia", *SEGMENTO)
    .agg(F.sum("valor_total").alias("receita_total"), F.sum("valor_energia").alias("receita_energia"),
         F.sum("tributos").alias("tributos"), F.sum("consumo_kwh").alias("consumo_kwh"),
         F.count("*").alias("faturas"))
    .withColumns({c: dec(c) for c in ["receita_total", "receita_energia", "tributos", "consumo_kwh"]})
    .withColumn("tarifa_media", F.try_divide("receita_energia", "consumo_kwh").cast("decimal(14,6)")))

In [0]:
# agg_preco_volume: P1 × P2 por segmento. Segmento sem um dos períodos: tudo vira efeito-volume
periodo = F.when(F.col("mes_referencia").between(*P1), "P1").when(F.col("mes_referencia").between(*P2), "P2")
pv = (receita.withColumn("periodo", periodo).filter(F.col("periodo").isNotNull())
    .groupBy(*SEGMENTO).pivot("periodo", ["P1", "P2"])
    .agg(F.sum("receita_energia").alias("receita"), F.sum("consumo_kwh").alias("kwh"))
    .fillna(0, subset=["P1_receita", "P1_kwh", "P2_receita", "P2_kwh"]))   # sem faturas no período = 0 conhecido

t1 = F.try_divide("P1_receita", "P1_kwh")
t2 = F.try_divide("P2_receita", "P2_kwh")
pv = (pv.withColumns({
        "tarifa_p1": t1.cast("decimal(14,6)"), "tarifa_p2": t2.cast("decimal(14,6)"),
        "variacao_receita": (F.col("P2_receita") - F.col("P1_receita")).cast("decimal(14,2)"),
        "efeito_volume": ((F.col("P2_kwh") - F.col("P1_kwh")) * F.coalesce(t1, t2)).cast("decimal(14,2)")})
    .withColumn("efeito_preco", (F.col("variacao_receita") - F.col("efeito_volume")).cast("decimal(14,2)"))
    .select(*SEGMENTO,
            *[F.col(a).cast("decimal(14,2)").alias(b) for a, b in [
                ("P1_receita", "receita_energia_p1"), ("P2_receita", "receita_energia_p2"),
                ("P1_kwh", "kwh_p1"), ("P2_kwh", "kwh_p2")]],
            "tarifa_p1", "tarifa_p2", "variacao_receita", "efeito_volume", "efeito_preco"))

In [0]:
# Checagens: falham ANTES de gravar
checar_unicidade(receita, ["mes_referencia", *SEGMENTO], "agg_receita_mensal")
checar_sem_nulos(receita, ["mes_referencia", *SEGMENTO, "receita_total", "receita_energia"], "agg_receita_mensal")
checar_unicidade(pv, SEGMENTO, "agg_preco_volume")

# Reconciliação com a Silver (nenhuma fatura ou leitura perdida no caminho)
g = receita.agg(F.sum("receita_total").alias("r"), F.sum("consumo_kwh").alias("k"), F.sum("faturas").alias("n")).first()
s = ft.agg(F.sum("valor_total").alias("r"), F.count("*").alias("n")).first()
k = lt.join(ft.select("id_uc", "mes_referencia"), ["id_uc", "mes_referencia"]).agg(F.sum("consumo_kwh")).first()[0]
assert (g["r"], g["n"], g["k"]) == (s["r"], s["n"], k), f"reconciliação não fecha: gold={tuple(g)} silver={tuple(s)} kwh={k}"

# Preço × volume fecha com a variação da receita de energia entre P1 e P2
tot = pv.agg(*[F.sum(c).alias(c) for c in ["receita_energia_p1", "receita_energia_p2", "kwh_p1", "kwh_p2",
                                           "variacao_receita", "efeito_volume", "efeito_preco"]]).first()
assert tot["efeito_volume"] + tot["efeito_preco"] == tot["variacao_receita"], "preço + volume ≠ variação"
pct = lambda a, b: f"{(b / a - 1) * 100:+.1f}%"
print(f"receita de energia P1→P2 {pct(tot['receita_energia_p1'], tot['receita_energia_p2'])} | "
      f"kWh {pct(tot['kwh_p1'], tot['kwh_p2'])} | efeito-volume R$ {tot['efeito_volume']:,.2f} | "
      f"efeito-preço R$ {tot['efeito_preco']:,.2f}")

In [0]:
# Gravar (overwrite: reexecutável) e mostrar onde está a queda de volume
for tabela, df in [("agg_receita_mensal", receita), ("agg_preco_volume", pv)]:
    (df.withColumn("_atualizado_em", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOGO}.gold.{tabela}"))
    print(f"gold.{tabela}: {spark.table(f'{CATALOGO}.gold.{tabela}').count():,} linhas")

display(spark.table(f"{CATALOGO}.gold.agg_preco_volume").orderBy("efeito_volume").limit(20))